# Problem 4 — Site Allow/Deny-List Agent (Hybrid: Cisco Umbrella + Infoblox)

> *GIS requests a site to be whitelisted or blacklisted. Automate the end-to-end
> fulfillment of this operation.*

**Hybrid scope:** many orgs run **two** DNS control points — **Cisco Umbrella** for cloud /
roaming / off-network resolution, and **Infoblox** (NIOS) for on-prem / internal
resolution. A block or allow is only real if it's enforced at **both**, so this agent fans
each request out across both layers and verifies each independently.

- **Cisco Umbrella** — Destination Lists (`access:"block"` / `access:"allow"`).
- **Infoblox NIOS** — Response Policy Zone (RPZ) rules via the WAPI: a `record:rpz:cname`
  with `canonical=""` is a **Block (NXDOMAIN)** rule; `canonical == name` is a
  **Passthru (allow)** rule.

## How indicator types map across the two layers
| Indicator | Umbrella | Infoblox RPZ | Net result |
|-----------|----------|--------------|------------|
| Domain | block/allow list | block (NXDOMAIN) / passthru | enforced on **both** |
| URL | full URL (block list, via proxy) | domain-level RPZ (path-blind) | both, different granularity |
| IPv4 (allow) | allow list (IPv4 ok) | n/a (RPZ is domain-based) | Umbrella only |
| IPv4 (deny) | n/a (Umbrella IPs are allow-only) | n/a | **escalated to the firewall** |

So a **full-URL block** is enforced precisely by Umbrella's proxy and at the domain level
by Infoblox; a **bare-IP block** isn't a DNS job at all → the agent escalates it (that's
NGFW/ASA territory).

## What this agent does
A **LangGraph** workflow:
1. **Intake** — read the ServiceNow request (indicator, allow/deny, scope, duration).
2. **Triage** — classify the destination, pull reputation, apply guardrails.
3. **Plan** — decide block vs allow + a comment (the per-layer fan-out is deterministic).
4. **Approve** — pause for a human (always for whitelist; always when reputation is bad).
5. **Apply** — fan out to **every supported layer**, then read each back to **verify**;
   close the ticket with per-layer status.

```
                                         ┌─► Umbrella  (cloud DNS/SWG)  destinationlists
 SNOW request ─► triage ─► plan ─► [approve] ─┤
  (allow/deny)   (classify (access  interrupt) └─► Infoblox (on-prem DNS)  record:rpz:cname
                  +guard)  +comment)              │
                                          verify each layer ─► close (per-layer status)
```

**Guardrails that override the model**
- Never block critical infrastructure (`microsoft.com`, Windows Update, identity
  providers, your own domains) — refused.
- Never auto-allow a known-malicious indicator — escalated.
- DNS-layer-impossible requests (IP block, URL allow) — escalated with a clear reason.

Everything runs offline against **mocks that mirror the Umbrella and Infoblox APIs**; real
API + CLI usage for both is in §9.

## 0. Setup
Runs with **no credentials** — ServiceNow, threat-intel, Umbrella, and Infoblox are all
simulated. Set an LLM key in `.env` for real planning; otherwise a deterministic stub runs
the flow.

In [ ]:
import os, re, json, datetime as dt
from typing import TypedDict, Literal, Optional
from urllib.parse import urlparse
from pydantic import BaseModel, Field
import pandas as pd

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import SystemMessage, HumanMessage

try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
print("Imports OK")

In [ ]:
def get_secret(name, default=None):
    v = os.getenv(name)
    if v:
        return v
    try:
        with open("config.local.json") as f:
            return json.load(f).get(name, default)
    except FileNotFoundError:
        return default

AUDIT = []
def audit(action, target, detail=None, actor="dns-allowdeny-agent"):
    rec = {"ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
           "actor": actor, "action": action, "target": target, "detail": detail or {}}
    AUDIT.append(rec)
    print(f"  [audit] {action:26s} {target}  {detail or ''}")
    return rec

def build_llm(temperature=0, model=None):
    provider = os.getenv("LLM_PROVIDER", "anthropic").lower()
    model = model or os.getenv("LLM_MODEL")
    try:
        if provider == "anthropic":
            key = get_secret("ANTHROPIC_API_KEY")
            if not key:
                return None
            from langchain_anthropic import ChatAnthropic
            return ChatAnthropic(model=model or "claude-sonnet-4-6",
                                 temperature=temperature, api_key=key, max_tokens=2048)
        if provider == "azure_openai":
            key = get_secret("AZURE_OPENAI_API_KEY")
            if not key:
                return None
            from langchain_openai import AzureChatOpenAI
            return AzureChatOpenAI(
                azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1"),
                api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21"),
                azure_endpoint=get_secret("AZURE_OPENAI_ENDPOINT"),
                api_key=key, temperature=temperature)
        if provider == "openai":
            key = get_secret("OPENAI_API_KEY")
            if not key:
                return None
            from langchain_openai import ChatOpenAI
            return ChatOpenAI(model=model or "gpt-4.1", api_key=key, temperature=temperature)
    except Exception as e:
        print("build_llm: offline stub ->", e)
        return None
    return None

LLM = build_llm()
LLM_AVAILABLE = LLM is not None
print("LLM:", "LIVE (" + os.getenv("LLM_PROVIDER", "anthropic") + ")" if LLM_AVAILABLE
      else "OFFLINE deterministic stub (set an API key in .env to enable real reasoning)")

def llm_structured(system, user, schema, demo):
    if LLM is None:
        return demo
    return LLM.with_structured_output(schema).invoke(
        [SystemMessage(content=system), HumanMessage(content=user)])

## 1. Cisco Umbrella client (mock that mirrors the Destination Lists API)

Mirrors the unified Umbrella API (`https://api.umbrella.com/policies/v2/...`). Going live is
swapping the mock for the `requests` calls in §9 — the agent code does not change.

In [ ]:
class MockUmbrella:
    """In-memory stand-in for the Umbrella Destination Lists API."""
    def __init__(self):
        self._lists = {
            11111: {"id": 11111, "access": "block", "name": "Global Block List", "isGlobal": True},
            22222: {"id": 22222, "access": "allow", "name": "Global Allow List", "isGlobal": True},
            33333: {"id": 33333, "access": "block", "name": "Finance-Workstations Block", "isGlobal": False},
            44444: {"id": 44444, "access": "allow", "name": "Finance-Workstations Allow", "isGlobal": False},
        }
        self._dests = {lid: [] for lid in self._lists}
        self._seq = 0

    def list_destination_lists(self):
        return {"status": {"code": 200}, "data": [
            {**l, "meta": {"destinationCount": len(self._dests[l["id"]])}}
            for l in self._lists.values()]}
        # === REAL API === GET https://api.umbrella.com/policies/v2/destinationlists

    def add_destinations(self, list_id, items):
        for it in items:
            self._seq += 1
            self._dests[list_id].append({"id": self._seq, "destination": it["destination"],
                                         "comment": it.get("comment", "")})
        audit("umbrella.add_destination", items[0]["destination"],
              {"list": self._lists[list_id]["name"]})
        return {"status": {"code": 200}, "data": {"id": list_id}}
        # === REAL API === POST .../policies/v2/destinationlists/{id}/destinations
        #   body: [{"destination": "...", "comment": "..."}]

    def get_destinations(self, list_id):
        return {"status": {"code": 200}, "data": list(self._dests[list_id])}
        # === REAL API === GET .../destinationlists/{id}/destinations

UMBRELLA = MockUmbrella()
print("Umbrella ready | lists:", [l["name"] for l in UMBRELLA.list_destination_lists()["data"]])

## 2. ServiceNow intake, threat-intel, and guardrails

The demo requests exercise the full cross-layer matrix — a URL block (Umbrella keeps the
path, Infoblox blocks the domain) and an IP block (no DNS layer can do it → escalated).

In [ ]:
REQUESTS = [
    {"number": "REQ0101", "indicator": "malware-c2.example",            "action": "deny",
     "scope": "enterprise", "duration_days": None, "requested_by": "soc.analyst",
     "justification": "Confirmed C2 from incident INC0456"},
    {"number": "REQ0102", "indicator": "vendor-portal.partner.com",     "action": "allow",
     "scope": "Finance-Workstations", "duration_days": 90, "requested_by": "ap.manager",
     "justification": "New AP vendor portal blocked by content category"},
    {"number": "REQ0103", "indicator": "microsoft.com",                 "action": "deny",
     "scope": "enterprise", "duration_days": None, "requested_by": "helpdesk.t1",
     "justification": "User thinks a 'microsoft popup' is malware"},
    {"number": "REQ0104", "indicator": "free-prizes-login.ru",          "action": "allow",
     "scope": "enterprise", "duration_days": 30, "requested_by": "marketing.intern",
     "justification": "Campaign landing page"},
    {"number": "REQ0105", "indicator": "https://bad.example/login.php", "action": "deny",
     "scope": "enterprise", "duration_days": None, "requested_by": "soc.analyst",
     "justification": "Phishing kit URL from user report"},
    {"number": "REQ0106", "indicator": "203.0.113.66",                  "action": "deny",
     "scope": "enterprise", "duration_days": None, "requested_by": "soc.analyst",
     "justification": "Scanning our perimeter"},
    {"number": "REQ0107", "indicator": "198.51.100.10",                 "action": "allow",
     "scope": "enterprise", "duration_days": 30, "requested_by": "neteng",
     "justification": "Partner SFTP source IP"},
]

def snow_fetch_requests():
    return list(REQUESTS)
    # === REAL API === GET {SNOW}/api/now/table/sc_req_item?sysparm_query=cat_item=allowdeny^active=true

def snow_update(number, state, work_notes=""):
    audit("servicenow.update", number, {"state": state, "notes": work_notes[:80]})
    return {"number": number, "state": state}
    # === REAL API === PATCH {SNOW}/api/now/table/sc_req_item/{sys_id}

_REPUTATION = {
    "malware-c2.example": {"verdict": "malicious", "score": 92},
    "free-prizes-login.ru": {"verdict": "malicious", "score": 88},
    "vendor-portal.partner.com": {"verdict": "clean", "score": 3},
    "bad.example": {"verdict": "malicious", "score": 90},
    "203.0.113.66": {"verdict": "suspicious", "score": 61},
    "198.51.100.10": {"verdict": "clean", "score": 5},
    "microsoft.com": {"verdict": "clean", "score": 0},
}
def threat_intel(value):
    return _REPUTATION.get(value, {"verdict": "unknown", "score": 50})
    # === REAL API === Cisco Talos / Umbrella Investigate / Infoblox Threat Intel / VirusTotal

CRITICAL_NEVER_BLOCK = {
    "microsoft.com", "windowsupdate.com", "office.com", "office365.com",
    "login.microsoftonline.com", "azure.com", "contoso.com", "okta.com",
    "google.com", "apple.com", "amazonaws.com", "cisco.com", "infoblox.com",
}

In [ ]:
IPV4 = re.compile(r"^\d{1,3}(\.\d{1,3}){3}$")

def classify_destination(raw):
    """Destination type: ipv4 | url | domain. URLs are kept whole for Umbrella."""
    raw = raw.strip()
    if IPV4.match(raw):
        return "ipv4", raw
    if "://" in raw or "/" in raw:
        return "url", raw
    return "domain", raw

def host_of(dest, dtype):
    if dtype == "url":
        return urlparse(dest if "://" in dest else "http://" + dest).hostname or dest
    return dest

def registrable(domain):
    parts = domain.split(".")
    return ".".join(parts[-2:]) if len(parts) >= 2 else domain

def guardrail_verdict(action, dtype, dest):
    """PROCEED / REFUSE / ESCALATE — code-enforced, overrides the LLM."""
    rep = threat_intel(host_of(dest, dtype))
    if action == "deny" and dtype in ("domain", "url"):
        host = host_of(dest, dtype)
        if host in CRITICAL_NEVER_BLOCK or registrable(host) in CRITICAL_NEVER_BLOCK:
            return "REFUSE", f"{host} is critical infrastructure — blocking is prohibited", rep
    if action == "allow" and rep["verdict"] == "malicious":
        return "ESCALATE", f"cannot auto-allow a known-malicious indicator (rep={rep['score']})", rep
    if action == "deny" and dtype == "ipv4":
        return ("ESCALATE", "blocking a bare IP is not a DNS function — route to the "
                "firewall/NGFW (e.g., Cisco ASA)", rep)
    if action == "allow" and dtype == "url":
        return ("ESCALATE", "a URL allow can't be expressed at the DNS layer — submit the "
                "domain instead", rep)
    return "PROCEED", "passed guardrails", rep

## 3. Hybrid enforcement layers (Umbrella + Infoblox)

Each layer is an adapter with the same shape — `supports(plan)`, `apply(plan)`,
`verify(plan)` — registered in `ENFORCEMENT_LAYERS`. The apply node simply iterates the
registry, so adding a third control point later (e.g., a Cisco ASA object-group for IPs) is
just one more adapter — no change to the graph.

In [ ]:
class MockInfoblox:
    """In-memory stand-in for NIOS WAPI record:rpz:cname (Response Policy Zone).
    canonical "" -> Block (NXDOMAIN); canonical == name -> Passthru (allow)."""
    RPZ_ZONE = os.getenv("INFOBLOX_RPZ_ZONE", "rpz.corp.local")

    def __init__(self):
        self._recs = {}   # domain -> record
        self._seq = 0

    def create_rpz_cname(self, domain, rule, comment=""):
        canonical = domain if rule == "passthru" else ""        # "" == Block (NXDOMAIN)
        self._seq += 1
        ref = f"record:rpz:cname/{self._seq}:{domain}.{self.RPZ_ZONE}"
        kind = "Passthru (allow)" if rule == "passthru" else "Block (NXDOMAIN)"
        self._recs[domain] = {"_ref": ref, "name": domain, "canonical": canonical,
                              "rp_zone": self.RPZ_ZONE, "comment": comment, "rule": kind}
        audit("infoblox.rpz_create", domain, {"rule": kind, "zone": self.RPZ_ZONE})
        return {"ok": True, "ref": ref, "rule": kind}
        # === REAL API === POST https://<grid-master>/wapi/v2.13/record:rpz:cname
        #   body: {"name": domain, "canonical": "" (block) | domain (passthru),
        #          "rp_zone": RPZ_ZONE}  ; auth: HTTP Basic (admin).

    def find_rpz(self, domain):
        return self._recs.get(domain)
        # === REAL API === GET /wapi/v2.13/record:rpz:cname?name={domain}&rp_zone={zone}

    def delete_rpz(self, domain):
        r = self._recs.pop(domain, None)
        if r:
            audit("infoblox.rpz_delete", domain)
        return {"ok": r is not None}
        # === REAL API === DELETE /wapi/v2.13/{_ref}

INFOBLOX = MockInfoblox()
print("Infoblox ready | RPZ zone:", INFOBLOX.RPZ_ZONE)

In [ ]:
def pick_list(access, scope):
    """Umbrella: choose a destination list by access + scope (scoped if present, else global)."""
    lists = UMBRELLA.list_destination_lists()["data"]
    scoped = [l for l in lists if l["access"] == access and not l["isGlobal"]
              and scope.lower() in l["name"].lower()]
    return scoped[0] if scoped else next(
        l for l in lists if l["access"] == access and l["isGlobal"])

class UmbrellaLayer:
    name = "umbrella"
    def supports(self, p):
        return True, "cloud DNS / SWG"           # guardrails already excluded the impossible combos
    def apply(self, p):
        dl = pick_list(p["access"], p["scope"])
        ok = UMBRELLA.add_destinations(
            dl["id"], [{"destination": p["destination"], "comment": p["comment"]}]
        )["status"]["code"] == 200
        return {"ok": ok, "where": dl["name"]}
    def verify(self, p):
        dl = pick_list(p["access"], p["scope"])
        return any(d["destination"] == p["destination"]
                   for d in UMBRELLA.get_destinations(dl["id"])["data"])

class InfobloxLayer:
    name = "infoblox"
    def supports(self, p):
        if p["dest_type"] == "ipv4":
            return False, "RPZ is domain-based (on-prem DNS does not enforce IPs)"
        return True, ("on-prem DNS RPZ" +
                      (" — domain-level for URLs" if p["dest_type"] == "url" else ""))
    def _domain(self, p):
        return host_of(p["destination"], p["dest_type"])
    def apply(self, p):
        rule = "passthru" if p["access"] == "allow" else "block"
        r = INFOBLOX.create_rpz_cname(self._domain(p), rule, comment=p["comment"])
        return {"ok": r["ok"], "where": f"{r['rule']} @ {INFOBLOX.RPZ_ZONE}"}
    def verify(self, p):
        return INFOBLOX.find_rpz(self._domain(p)) is not None

ENFORCEMENT_LAYERS = [UmbrellaLayer(), InfobloxLayer()]
print("Enforcement layers:", [l.name for l in ENFORCEMENT_LAYERS])

## 4. Agent state + plan schema

In [ ]:
class DestinationPlan(BaseModel):
    request_id: str
    destination: str                       # domain / full URL / IPv4 as submitted
    dest_type: Literal["domain", "url", "ipv4"]
    access: Literal["block", "allow"]      # block = blacklist, allow = whitelist
    scope: str
    comment: str
    verdict: Literal["PROCEED", "REFUSE", "ESCALATE"]
    rationale: str

class AgentState(TypedDict):
    requests: list[dict]
    triaged: list[dict]
    plans: list[dict]
    approved: list[str]
    results: list[dict]
    report: dict

TODAY = dt.date(2026, 6, 2)

def rule_plan(t) -> DestinationPlan:
    """Deterministic offline planner (layer-agnostic; fan-out happens at apply time)."""
    access = "block" if t["action"] == "deny" else "allow"
    comment = f"{t['number']}: {t['justification']}"
    if access == "allow" and t["duration_days"]:
        exp = (TODAY + dt.timedelta(days=t["duration_days"])).isoformat()
        comment += f" | expires {exp} (reconcile job removes — no native TTL)"
    return DestinationPlan(request_id=t["number"], destination=t["dest_value"],
                           dest_type=t["dest_type"], access=access, scope=t["scope"],
                           comment=comment, verdict=t["guardrail"],
                           rationale=t["guardrail_reason"])

def planned_layers(p):
    return [l.name for l in ENFORCEMENT_LAYERS if l.supports(p)[0]]

## 5. Graph nodes

In [ ]:
def n_intake(state: AgentState) -> dict:
    reqs = snow_fetch_requests()
    audit("intake.requests", "servicenow", {"count": len(reqs)})
    return {"requests": reqs}

def n_triage(state: AgentState) -> dict:
    triaged = []
    for req in state["requests"]:
        dtype, dval = classify_destination(req["indicator"])
        verdict, reason, rep = guardrail_verdict(req["action"], dtype, dval)
        triaged.append({**req, "dest_type": dtype, "dest_value": dval, "reputation": rep,
                        "guardrail": verdict, "guardrail_reason": reason})
        print(f"[triage] {req['number']} {req['action']:5s} {req['indicator']:30s} "
              f"type={dtype:6s} rep={rep['verdict']:9s} -> {verdict}")
    return {"triaged": triaged}

In [ ]:
PLAN_SYS = (
    "You fulfill DNS allow/deny requests enforced on BOTH Cisco Umbrella (cloud) and "
    "Infoblox RPZ (on-prem). Decide access ('block' for deny, 'allow' for allow) and write "
    "a clear comment (include any expiry). The per-layer fan-out is handled automatically. "
    "Respect the guardrail verdict: if REFUSE or ESCALATE, return it and do not enforce."
)

def n_plan(state: AgentState) -> dict:
    plans = []
    for t in state["triaged"]:
        demo = rule_plan(t)
        user = json.dumps({k: t[k] for k in ("number", "indicator", "dest_type", "dest_value",
                          "action", "scope", "duration_days", "reputation",
                          "guardrail", "guardrail_reason")}, default=str)
        plan = llm_structured(PLAN_SYS, "Plan this request:\n" + user, DestinationPlan, demo)
        if t["guardrail"] != "PROCEED":
            plan.verdict = t["guardrail"]
        p = plan.model_dump(); p["_req"] = t
        plans.append(p)
        tag = f"{p['access']} via {planned_layers(p)}" if p["verdict"] == "PROCEED" else "-"
        print(f"[plan]   {p['request_id']} {p['verdict']:8s} {tag}")
    return {"plans": plans}

In [ ]:
def n_approval(state: AgentState) -> dict:
    actionable = [p for p in state["plans"] if p["verdict"] == "PROCEED"]
    for p in state["plans"]:
        if p["verdict"] == "REFUSE":
            snow_update(p["request_id"], "closed_rejected", p["rationale"])
        elif p["verdict"] == "ESCALATE":
            snow_update(p["request_id"], "escalated", p["rationale"])
    if not actionable:
        return {"approved": []}
    decision = interrupt({
        "type": "approve_dns_changes",
        "summary": f"{len(actionable)} change(s) ready across the hybrid DNS layers",
        "changes": [{"request": p["request_id"], "access": p["access"],
                     "destination": p["destination"], "layers": planned_layers(p),
                     "reputation": p["_req"]["reputation"]["verdict"]} for p in actionable],
        "note": "Allow-list entries weaken filtering — confirm before approving.",
        "instructions": "resume with {'approved': [request ids]} or {'approved': 'ALL'}",
    })
    approved = decision.get("approved", []) if isinstance(decision, dict) else []
    if approved == "ALL":
        approved = [p["request_id"] for p in actionable]
    audit("human.approval", "dns-batch", {"approved": approved})
    return {"approved": approved}

In [ ]:
def n_apply(state: AgentState) -> dict:
    by_id = {p["request_id"]: p for p in state["plans"]}
    results = []
    for rid in state["approved"]:
        p = by_id[rid]
        layer_results = []
        for layer in ENFORCEMENT_LAYERS:
            ok_support, why = layer.supports(p)
            if not ok_support:
                layer_results.append({"layer": layer.name, "status": "skipped", "detail": why})
                continue
            out = layer.apply(p)
            verified = layer.verify(p) if out["ok"] else False
            layer_results.append({"layer": layer.name, "detail": out.get("where", ""),
                                  "status": "enforced" if (out["ok"] and verified) else "FAILED"})
        applied = [l for l in layer_results if l["status"] == "enforced"]
        failed = [l for l in layer_results if l["status"] == "FAILED"]
        ok = bool(applied) and not failed
        snow_update(rid, "closed_complete" if ok else "work_in_progress",
                    "; ".join(f"{l['layer']}={l['status']}" for l in layer_results))
        results.append({"request_id": rid, "destination": p["destination"],
                        "access": p["access"], "layers": layer_results, "ok": ok})
        print(f"[apply]  {rid} {p['access']:5s} {p['destination']:28s} " +
              " | ".join(f"{l['layer']}:{l['status']}" for l in layer_results))
    return {"results": results}

def n_report(state: AgentState) -> dict:
    enforced = [r for r in state["results"] if r["ok"]]
    def cnt(layer):
        return sum(1 for r in state["results"] for l in r["layers"]
                   if l["layer"] == layer and l["status"] == "enforced")
    report = {"requests": len(state["requests"]),
              "enforced": [r["request_id"] for r in enforced],
              "umbrella_changes": cnt("umbrella"), "infoblox_changes": cnt("infoblox"),
              "blacklisted": [r["destination"] for r in enforced if r["access"] == "block"],
              "whitelisted": [r["destination"] for r in enforced if r["access"] == "allow"],
              "refused": [p["request_id"] for p in state["plans"] if p["verdict"] == "REFUSE"],
              "escalated": [p["request_id"] for p in state["plans"] if p["verdict"] == "ESCALATE"]}
    print(f"[report] {report}")
    return {"report": report}

## 6. Build & compile the graph

In [ ]:
g = StateGraph(AgentState)
for name, fn in [("intake", n_intake), ("triage", n_triage), ("plan", n_plan),
                 ("approval", n_approval), ("apply", n_apply), ("report", n_report)]:
    g.add_node(name, fn)
g.add_edge(START, "intake")
g.add_edge("intake", "triage")
g.add_edge("triage", "plan")
g.add_edge("plan", "approval")
g.add_edge("approval", "apply")
g.add_edge("apply", "report")
g.add_edge("report", END)
app = g.compile(checkpointer=MemorySaver())
print("Graph compiled:", list(app.get_graph().nodes))

## 7. Run it

In [ ]:
AUDIT.clear()
config = {"configurable": {"thread_id": "dns-run-2026-06-02"}}
init = {"requests": [], "triaged": [], "plans": [], "approved": [], "results": [], "report": {}}
result = app.invoke(init, config)

print("\n=== PAUSED FOR APPROVAL ===")
intr = result["__interrupt__"][0].value
print(intr["summary"], "--", intr["note"])
for c in intr["changes"]:
    print(f"  - {c['request']}  {c['access']:5s} {c['destination']:28s} "
          f"layers={c['layers']}  rep={c['reputation']}")

In [ ]:
final = app.invoke(Command(resume={"approved": "ALL"}), config)
print("\n=== FINAL REPORT ===")
print(json.dumps(final["report"], indent=2))

## 8. Results + audit trail

In [ ]:
rows = []
for p in final["plans"]:
    res = next((r for r in final["results"] if r["request_id"] == p["request_id"]), None)
    layers = ", ".join(f"{l['layer']}:{l['status']}" for l in res["layers"]) if res else "-"
    rows.append({"request": p["request_id"], "action": p["_req"]["action"],
                 "type": p["dest_type"], "destination": p["destination"],
                 "verdict": p["verdict"], "layers": layers, "rationale": p["rationale"]})
display(pd.DataFrame(rows))
print()
display(pd.DataFrame(AUDIT)[["action", "target", "detail"]])

## 9. Using the real APIs / CLI

Both layers are plain REST, so the mocks swap out for `requests`/`curl`.

### Cisco Umbrella (Destination Lists) — OAuth2 client-credentials
```python
import requests
from requests.auth import HTTPBasicAuth
BASE = "https://api.umbrella.com"
tok = requests.post(f"{BASE}/auth/v2/token",
                    auth=HTTPBasicAuth(UMBRELLA_API_KEY, UMBRELLA_API_SECRET)).json()
H = {"Authorization": f"Bearer {tok['access_token']}", "Content-Type": "application/json"}
lists = requests.get(f"{BASE}/policies/v2/destinationlists", headers=H).json()["data"]
block_id = next(l["id"] for l in lists if l["access"] == "block" and l["isGlobal"])
requests.post(f"{BASE}/policies/v2/destinationlists/{block_id}/destinations", headers=H,
              json=[{"destination": "malware-c2.example", "comment": "ServiceNow REQ0101"}])
```

### Infoblox NIOS (RPZ via WAPI) — HTTP Basic to the Grid Master
```python
import requests
GM = "https://192.0.2.10/wapi/v2.13"               # Grid Master + WAPI version
AUTH = ("admin", INFOBLOX_PASSWORD)
ZONE = "rpz.corp.local"                            # an existing Response Policy Zone

# Blacklist a domain  -> Block (NXDOMAIN): canonical = ""
requests.post(f"{GM}/record:rpz:cname", auth=AUTH, verify=False,
    json={"name": "malware-c2.example", "canonical": "", "rp_zone": ZONE,
          "comment": "ServiceNow REQ0101"})

# Whitelist a domain  -> Passthru (allow): canonical == name
requests.post(f"{GM}/record:rpz:cname", auth=AUTH, verify=False,
    json={"name": "vendor-portal.partner.com", "canonical": "vendor-portal.partner.com",
          "rp_zone": ZONE})

# Verify / remove
ref = requests.get(f"{GM}/record:rpz:cname",
                   params={"name": "malware-c2.example", "rp_zone": ZONE},
                   auth=AUTH, verify=False).json()[0]["_ref"]
requests.delete(f"{GM}/{ref}", auth=AUTH, verify=False)
```

### CLI — `curl`
```bash
# Umbrella token + block
TOKEN=$(curl -s -u "$UMBRELLA_API_KEY:$UMBRELLA_API_SECRET" -X POST \
  https://api.umbrella.com/auth/v2/token | jq -r .access_token)
curl -s -X POST https://api.umbrella.com/policies/v2/destinationlists/11111/destinations \
  -H "Authorization: Bearer $TOKEN" -H "Content-Type: application/json" \
  -d '[{"destination":"malware-c2.example","comment":"REQ0101"}]'

# Infoblox RPZ block (NXDOMAIN)
curl -k -u "admin:$INFOBLOX_PASSWORD" -X POST \
  "https://192.0.2.10/wapi/v2.13/record:rpz:cname" \
  -H "Content-Type: application/json" \
  -d '{"name":"malware-c2.example","canonical":"","rp_zone":"rpz.corp.local"}'
```

## 10. Productionizing the hybrid
- **Enforce on both, verify on both.** A block that lands on Umbrella but fails on Infoblox
  (or vice-versa) is a coverage gap; the agent marks the request incomplete until both
  succeed (or a layer legitimately skips, e.g., Infoblox on an IP).
- **The adapter registry is the extension point.** Add Cisco ASA (object-group + ACL) as a
  third layer to actually fulfill IP blocks, or a SWG for path-level URL allow — each is one
  more class in `ENFORCEMENT_LAYERS`, no graph change.
- **Granularity differs by layer:** Umbrella enforces the full URL (proxy); Infoblox RPZ is
  domain-level. Make that explicit on the ticket so reviewers know a URL block also blocks
  the domain on-prem.
- **No native expiry on either** — store the expiry in the comment and run a reconcile job
  that removes both the Umbrella destination and the Infoblox RPZ record past their date.
- **Idempotency + least privilege:** check before create on each layer so re-runs don't
  duplicate; an Umbrella key scoped to Policies and an Infoblox admin scoped to the RPZ
  zone; key entries by request id for clean rollback.